In [0]:
from pyspark.sql.functions import current_timestamp

In [0]:
%python
# Define our environment variables
catalog = "medallion_trial"
schema = "bronze"
volume_path = f"/Volumes/{catalog}/{schema}/raw_landing_zone"

In [0]:
%python
# Create a dictionary of our files and what we want to name their Bronze tables
datasets = {
    "olist_orders_dataset.csv": "raw_orders",
    "olist_order_items_dataset.csv": "raw_order_items",
    "olist_customers_dataset.csv": "raw_customers",
    "olist_products_dataset.csv": "raw_products"
}

In [0]:
# Define a function to process each file
def load_csv_to_bronze(file_name, table_name):
    print(f"Ingesting {file_name} into {catalog}.{schema}.{table_name}...")
    
    # 1. Read the raw CSV exactly as-is
    # We set inferSchema to "false" so everything stays as a raw string for now
    df = spark.read.format("csv") \
        .option("header", "true") \
        .option("inferSchema", "false") \
        .load(f"{volume_path}/{file_name}")
        
    # 2. Add an ingestion timestamp (Best Practice!)
    df_with_metadata = df.withColumn("ingest_timestamp", current_timestamp())
        
    # 3. Write to Unity Catalog as a Delta Table
    df_with_metadata.write.format("delta") \
        .mode("overwrite") \
        .saveAsTable(f"{catalog}.{schema}.{table_name}")
        
    print(f"Success! {table_name} created.\n")

In [0]:
# Loop through our dictionary and run the function for each dataset
for file, table in datasets.items():
    load_csv_to_bronze(file, table)

In [0]:
%sql
SELECT * FROM medallion_trial.bronze.raw_orders LIMIT 5